In [1]:
from langgraph.graph import StateGraph, START, END, MessagesState

from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_qwq import ChatQwen

model = ChatQwen(
    model="qwen3.7-max",
)

def llm_node(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    response = model.invoke(messages)

    return {
        "messages": [response],
    }

builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

for chunk in graph.stream(
        {
            "messages":[HumanMessage(content="你好!")]
        },
    stream_mode=["values","messages"],
):
    print(chunk)


('values', {'messages': [HumanMessage(content='你好!', additional_kwargs={}, response_metadata={}, id='b1a825a4-27e4-4e7c-9309-6ba561fcd9f4')]})
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'dashscope'}, id='lc_run--019fd69a-d7ca-7ea0-9629-04a067cabe64', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'llm_node', 'langgraph_triggers': ('branch:to:llm_node',), 'langgraph_path': ('__pregel_pull', 'llm_node'), 'langgraph_checkpoint_ns': 'llm_node:f8746beb-66c8-9f8c-d059-1a278cf96313', 'checkpoint_ns': 'llm_node:f8746beb-66c8-9f8c-d059-1a278cf96313', 'ls_provider': 'openai', 'ls_model_name': 'qwen3.7-max', 'ls_model_type': 'chat', 'ls_temperature': None}))
('messages', (AIMessageChunk(content='', additional_kwargs={'reasoning_content': '用户'}, response_metadata={'model_provider': 'dashscope'}, id='lc_run--019fd69a-d7ca-7ea0-9629-04a067cabe64', tool_calls=[], invalid_tool_calls=[], tool_c